<!-- notebook-header -->
# Otimizacao para Machine Learning

**Modulo:** 00 - Matematica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Funcoes de custo, convexidade, GD, SGD, Momentum, AdaGrad, RMSprop, Adam e regularizacao.


# 0.8 Otimizacao para Machine Learning

**Tempo estimado:** 12-14 horas
**Pre-requisitos:** 0.4 (Derivadas), 0.5 (Integrais), 0.6-0.7 (Probabilidade)
**Proximo modulo:** 1.1 (Estatistica Descritiva)

---

## Indice

1. Introducao: ML como Otimizacao
2. Funcoes de Custo Comuns
3. Convexidade
4. Gradient Descent
5. Learning Rate
6. Batch vs SGD vs Mini-batch
7. SGD com Momentum
8. AdaGrad
9. RMSprop
10. Adam
11. Regularizacao (L1/L2)
12. Vanishing e Exploding Gradients
13. Aplicacao: Regressao Linear do Zero
14. Aplicacao: Regressao Logistica do Zero
15. Exercicios Praticos
16. Erros Comuns
17. Resumo e Mapa Conceitual

## Pre-requisitos e Fio Narrativo

### De onde viemos
| Conceito | Notebook | Como usamos aqui |
|----------|----------|-------------------|
| Derivadas parciais | 0.4 | Gradientes sao vetores de derivadas parciais |
| Regra da cadeia | 0.4 | Backpropagation = chain rule aplicada em camadas |
| Series e convergencia | 0.5 | Convergencia de algoritmos iterativos |
| MLE e cross-entropy | 0.7 | Funcoes de custo derivam de likelihood |
| Prior bayesiano | 0.7 | Regularizacao = prior sobre parametros |

### Para onde vamos
| Conceito daqui | Usado em | Como |
|----------------|----------|------|
| Gradient Descent | Todos os modelos | Todo modelo de ML usa alguma variante de GD |
| Adam optimizer | Deep Learning | Otimizador padrao para redes neurais |
| Regularizacao | Regressao, redes | Controle de overfitting em qualquer modelo |
| Vanishing gradients | RNNs, redes profundas | Problema central em deep learning |

### Fio narrativo
No modulo 0.7, vimos que treinar um modelo = maximizar likelihood dos dados. Isso equivale a **minimizar uma funcao de custo**. Mas como minimizar numericamente, quando nao existe solucao analitica? Este modulo ensina os **algoritmos de otimizacao** que fazem o ML funcionar na pratica. Comecamos com o basico (Gradient Descent) e construimos ate o Adam, o otimizador padrao em deep learning.

## Por que Otimizacao eh Fundamental em ML?

Quase todo algoritmo de ML segue o mesmo padrao:

1. **Definir modelo:** $f(x; w)$ com parametros $w$
2. **Definir custo:** $\mathcal{L}(w)$ que mede "quao ruim estao as predicoes"
3. **Minimizar custo:** encontrar $w^* = \arg\min_w \mathcal{L}(w)$

O passo 3 eh **otimizacao**. Sem ele, nenhum modelo aprende.

**Por que em ML:** Entender otimizacao permite diagnosticar problemas de treinamento (loss nao converge, overfitting, gradientes explodindo), escolher hiperparametros (learning rate, regularizacao), e selecionar o otimizador certo para cada problema. Sem esse conhecimento, treinar modelos vira tentativa-e-erro cego.

**Conexao com 0.7:** Lembre que maximizar $P(D|\theta)$ (MLE) equivale a minimizar $-\log P(D|\theta)$ (cross-entropy). Otimizacao eh a ponte entre teoria probabilistica e pratica computacional.

## 1. Introducao: ML como Problema de Otimizacao

### Analogia: Encontrar o Vale mais Fundo

Imagine que voce esta numa montanha coberta por neblina. Nao consegue ver o vale, mas sente a inclinacao do chao sob seus pes. O que faz? **Desce na direcao mais ingreme.** Isso eh Gradient Descent.

### Formulacao Formal

Todo algoritmo de ML resolve:

$$w^* = \arg\min_w \mathcal{L}(w) = \arg\min_w \frac{1}{n}\sum_{i=1}^n \ell\big(f(x_i; w),\; y_i\big) + \lambda \cdot R(w)$$

onde:
- $\ell$: funcao de perda por amostra (MSE, cross-entropy, hinge...)
- $R(w)$: termo de regularizacao (L1, L2)
- $\lambda$: forca da regularizacao

**Por que em ML:** Esta formulacao unifica regressao linear, logistica, SVMs, redes neurais -- todos sao casos especiais de escolher $f$, $\ell$ e $R$ diferentes.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print("Bibliotecas carregadas com sucesso")
print(f"NumPy: {np.__version__}")

Bibliotecas carregadas com sucesso
NumPy: 2.4.6


## 2. Funcoes de Custo Comuns

### Analogia: Notas numa Prova

Imagine que um professor precisa decidir como penalizar erros dos alunos:
- **MSE (Mean Squared Error):** penaliza erros grandes muito mais que pequenos (como um professor rigoroso que pune erros graves)
- **MAE (Mean Absolute Error):** penaliza todos os erros proporcionalmente (professor justo, sem pesos extras)
- **Huber:** combina os dois: justo para erros pequenos, rigoroso para grandes
- **Hinge Loss:** so penaliza se a resposta esta do lado errado da margem (SVM)

### Definicoes Formais

| Custo | Formula | Quando usar |
|-------|---------|-------------|
| MSE | $\frac{1}{n}\sum(y - \hat{y})^2$ | Regressao, dados sem outliers |
| MAE | $\frac{1}{n}\sum|y - \hat{y}|$ | Regressao com outliers |
| Huber | Quadratico se $|e|\leq\delta$, linear caso contrario | Regressao robusta |
| Cross-Entropy | $-\frac{1}{n}\sum[y\log\hat{y} + (1-y)\log(1-\hat{y})]$ | Classificacao |
| Hinge | $\frac{1}{n}\sum\max(0, 1 - y\hat{y})$ | SVM, margem maxima |

**Por que em ML:** A escolha da funcao de custo determina o comportamento do modelo. MSE maximiza likelihood gaussiano (conexao 0.7). Cross-entropy maximiza likelihood de Bernoulli. A funcao de custo codifica suas **premissas sobre o problema**.

**Conexao com 0.7:** MSE emerge naturalmente quando assumimos ruido gaussiano nos dados ($y = f(x) + \epsilon$, $\epsilon \sim \mathcal{N}(0, \sigma^2)$). Minimizar MSE = maximizar likelihood gaussiano.

In [2]:
y_true = 1.0
y_pred = np.linspace(-2, 3, 100)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
mse = (y_true - y_pred)**2
ax.plot(y_pred, mse, linewidth=2, color='blue')
ax.set_xlabel('Predicao')
ax.set_ylabel('Perda')
ax.set_title('MSE: quadratico')
ax.grid(alpha=0.3)
ax.axvline(y_true, color='red', linestyle='--', alpha=0.7)

ax = axes[0, 1]
mae = np.abs(y_true - y_pred)
ax.plot(y_pred, mae, linewidth=2, color='green')
ax.set_xlabel('Predicao')
ax.set_ylabel('Perda')
ax.set_title('MAE: linear (robusto)')
ax.grid(alpha=0.3)
ax.axvline(y_true, color='red', linestyle='--', alpha=0.7)

ax = axes[1, 0]
huber = np.where(np.abs(y_true - y_pred) <= 1, 0.5*(y_true - y_pred)**2, 
                 np.abs(y_true - y_pred) - 0.5)
ax.plot(y_pred, huber, linewidth=2, color='orange')
ax.plot(y_pred, mse, linewidth=1.5, alpha=0.5, color='blue', label='MSE')
ax.plot(y_pred, mae, linewidth=1.5, alpha=0.5, color='green', label='MAE')
ax.set_xlabel('Predicao')
ax.set_ylabel('Perda')
ax.set_title('Huber: hibrido')
ax.legend()
ax.grid(alpha=0.3)
ax.axvline(y_true, color='red', linestyle='--', alpha=0.7)

ax = axes[1, 1]
hinge = np.maximum(0, 1 - y_true * y_pred)
ax.plot(y_pred, hinge, linewidth=2, color='purple')
ax.set_xlabel('Predicao * y_true')
ax.set_ylabel('Perda')
ax.set_title('Hinge Loss: SVM')
ax.grid(alpha=0.3)
ax.axvline(1, color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

print('Observacoes:')
print('1. MSE penaliza erros grandes mais')
print('2. MAE penaliza linearmente (robusto a outliers)')
print('3. Huber eh hibrido')
print('4. Hinge para classificacao com margem')

Observacoes:
1. MSE penaliza erros grandes mais
2. MAE penaliza linearmente (robusto a outliers)
3. Huber eh hibrido
4. Hinge para classificacao com margem


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/1862982122.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- MSE cresce quadraticamente: erros de magnitude 2 custam 4x mais que erros de magnitude 1
- MAE cresce linearmente: todos os erros sao penalizados proporcionalmente
- Huber combina: quadratico perto de zero, linear longe (melhor dos dois mundos)
- Hinge Loss: custo zero quando predicao esta correta E com margem suficiente

**O que concluir:**
- Use MSE quando outliers sao raros e erros grandes sao realmente piores
- Use MAE/Huber quando ha outliers nos dados
- Use Cross-Entropy para classificacao (NUNCA MSE para classificacao!)
- A escolha do custo afeta quais parametros o modelo aprende

**Conexao com 0.4:** A derivada de cada custo determina como o gradiente se comporta. MSE tem gradiente proporcional ao erro ($2e$), MAE tem gradiente constante ($\pm 1$). Isso afeta a velocidade de convergencia.

## 3. Convexidade

### Analogia: Forma do Terreno

Imagine descer uma montanha:
- **Funcao convexa** = vale com formato de "U": nao importa onde voce comece, descendo sempre chega ao fundo (minimo global)
- **Funcao concava** = montanha invertida: so tem maximo, nao minimo
- **Funcao nao-convexa** = terreno irregular com vales e morros: voce pode ficar preso num vale raso (minimo local)

### Definicao Formal

$f$ eh **convexa** se, para quaisquer $x, y$ e $t \in [0,1]$:

$$f(tx + (1-t)y) \leq t\cdot f(x) + (1-t)\cdot f(y)$$

Geometricamente: qualquer corda entre dois pontos da curva fica **acima** da curva.

**Por que em ML:**
- Regressao linear + MSE = problema **convexo** (solucao unica garantida)
- Regressao logistica + cross-entropy = **convexo** (solucao unica)
- Redes neurais = **nao-convexo** (multiplos minimos locais, mas empiricamente funciona)

**Conexao com 0.4:** Para funcoes $C^2$, convexidade equivale a Hessiana (matriz de segundas derivadas) ser semi-definida positiva: $H \succeq 0$.

In [3]:
# Visualizar convexidade
x = np.linspace(-3, 3, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Convexa
y_convex = x**2
ax = axes[0]
ax.plot(x, y_convex, linewidth=2, color='green', label='Convexa: x^2')
ax.plot([-2, 2], [4, 4], 'r--', linewidth=2, alpha=0.7, label='Corda')
ax.scatter([-2, 2], [4, 4], s=100, color='red')
ax.fill_between(x[(x >= -2) & (x <= 2)], y_convex[(x >= -2) & (x <= 2)], 4, alpha=0.2, color='green')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Funcao Convexa: minimo unico')
ax.legend()
ax.grid(alpha=0.3)

# Concava
y_concave = -x**2
ax = axes[1]
ax.plot(x, y_concave, linewidth=2, color='red', label='Concava: -x^2')
ax.plot([-2, 2], [-4, -4], 'b--', linewidth=2, alpha=0.7, label='Corda')
ax.scatter([-2, 2], [-4, -4], s=100, color='blue')
ax.fill_between(x[(x >= -2) & (x <= 2)], y_concave[(x >= -2) & (x <= 2)], -4, alpha=0.2, color='red')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Funcao Concava: maximo unico')
ax.legend()
ax.grid(alpha=0.3)

# Nao-convexa
y_nonconvex = np.sin(x)
ax = axes[2]
ax.plot(x, y_nonconvex, linewidth=2, color='purple', label='Nao-convexa: sin(x)')
ax.scatter([np.pi/2, -np.pi/2], [1, -1], s=100, color='purple', label='Maximos/minimos locais')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Funcao Nao-convexa: multiplos minimos')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Diferenca:')
print('- Convexa: otimizacao garantida')
print('- Nao-convexa: pode ficar em minimos locais')
print('- Deep Learning: nao-convexo, mas pratico funciona')

Diferenca:
- Convexa: otimizacao garantida
- Nao-convexa: pode ficar em minimos locais
- Deep Learning: nao-convexo, mas pratico funciona


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/4095570196.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Funcao convexa ($x^2$): qualquer corda entre dois pontos fica acima da curva; area verde mostra isso
- Funcao concava ($-x^2$): corda fica abaixo; exatamente o oposto
- Funcao nao-convexa ($\sin(x)$): multiplos maximos e minimos locais

**O que concluir:**
- Problemas convexos sao "faceis" para otimizacao: GD sempre encontra o minimo global
- Problemas nao-convexos (redes neurais) sao "dificeis" na teoria, mas na pratica minimos locais frequentemente sao "bons o suficiente"
- Verificar convexidade ajuda a saber se seu otimizador vai encontrar a solucao correta

**Conexao com 0.4:** A segunda derivada ($f''(x) > 0$) indica convexidade local. Para funcoes multivariadas, precisamos da Hessiana positiva semi-definida.

## 4. Gradient Descent

### Analogia: Descer a Montanha na Neblina

Voce esta na montanha sem visibilidade. A unica informacao que tem eh a **inclinacao do chao** sob seus pes (o gradiente). Estrategia: a cada passo, descer na direcao mais ingreme. Repetir ate chegar num vale.

### Algoritmo

$$w_{t+1} = w_t - \eta \cdot \nabla\mathcal{L}(w_t)$$

onde:
- $w_t$: parametros na iteracao $t$
- $\eta$: **learning rate** (tamanho do passo)
- $\nabla\mathcal{L}$: gradiente da funcao de custo (direcao de maior aumento)
- O sinal negativo garante que descemos (oposto ao gradiente)

### Propriedades
- **Convergencia garantida** para funcoes convexas com $\eta$ adequado
- **Taxa exponencial:** cada iteracao reduz o erro por fator constante
- **Custo por iteracao:** $O(n \cdot d)$ onde $n$ = amostras, $d$ = dimensoes

**Por que em ML:** GD eh o algoritmo fundamental. Tudo que vem depois (SGD, Adam, etc.) sao variacoes que melhoram velocidade ou estabilidade do GD basico.

**Conexao com 0.4:** O gradiente $\nabla\mathcal{L}$ eh exatamente o vetor de derivadas parciais: $\nabla\mathcal{L} = \left(\frac{\partial\mathcal{L}}{\partial w_1}, \ldots, \frac{\partial\mathcal{L}}{\partial w_d}\right)$. Cada componente diz "quanto o custo muda se eu mexer neste parametro".

In [4]:
# Demonstrar GD numa parabola
def f(x):
    return (x - 2)**2

def df(x):
    return 2*(x - 2)

# GD
eta = 0.1
x = 5.0
history = [x]

for _ in range(20):
    grad = df(x)
    x = x - eta * grad
    history.append(x)

history = np.array(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafico da funcao e trajetoria
ax = ax1
x_range = np.linspace(0, 8, 200)
y_range = f(x_range)
ax.plot(x_range, y_range, linewidth=2, color='blue', label='f(x) = (x-2)^2')
ax.plot(history, f(history), 'ro-', linewidth=2, markersize=8, label='Trajetoria GD')
ax.scatter([2], [0], s=200, color='green', marker='*', label='Minimo verdadeiro')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Gradient Descent')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Convergencia
ax = ax2
losses = f(history)
ax.semilogy(losses, linewidth=2, color='red', marker='o')
ax.set_xlabel('Iteracao')
ax.set_ylabel('Perda (escala log)')
ax.set_title('Convergencia Exponencial')
ax.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print('Propriedade GD:')
print(f'Iteracao 0: x={history[0]:.3f}, f(x)={f(history[0]):.3f}')
print(f'Iteracao 10: x={history[10]:.3f}, f(x)={f(history[10]):.6f}')
print(f'Iteracao 20: x={history[20]:.3f}, f(x)={f(history[20]):.9f}')
print('Convergencia exponencial!')

Propriedade GD:
Iteracao 0: x=5.000, f(x)=9.000
Iteracao 10: x=2.322, f(x)=0.103763
Iteracao 20: x=2.035, f(x)=0.001196305
Convergencia exponencial!


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/1335365563.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Grafico esquerdo: a trajetoria de GD (pontos vermelhos) desce pela parabola ate o minimo
- Grafico direito: a loss cai exponencialmente (linha reta em escala log)
- Apos ~10 iteracoes, o algoritmo ja esta muito proximo do minimo verdadeiro

**O que concluir:**
- GD eh surpreendentemente eficiente para problemas convexos
- A convergencia exponencial significa: cada iteracao reduz o erro por fator constante (~$1 - 2\eta$)
- O algoritmo inteiro sao apenas 3 linhas de codigo: calcular gradiente, atualizar peso, repetir
- A simplicidade eh uma vantagem: facil de implementar, debugar e analisar

**Conexao com 0.5:** A convergencia exponencial lembra series geometricas: $a, ar, ar^2, \ldots$ com $|r| < 1$. A razao $r$ depende do learning rate e da curvatura da funcao.

## 5. Learning Rate: O Hiperparametro Mais Critico

### Analogia: Tamanho dos Passos na Descida

- **$\eta$ muito pequeno:** como dar passos de formiga na montanha; voce desce, mas leva horas
- **$\eta$ adequado:** passos firmes e seguros; desce rapido sem tropecar
- **$\eta$ muito grande:** como pular de penhasco em penhasco; pode pular o vale e cair do outro lado (divergencia!)

### Regras Praticas

| Learning Rate | Comportamento | Quando usar |
|---------------|--------------|-------------|
| $10^{-4}$ a $10^{-3}$ | Convergencia lenta, segura | Fine-tuning, modelos pre-treinados |
| $10^{-3}$ a $10^{-2}$ | Equilibrio velocidade/estabilidade | Treinamento geral |
| $> 10^{-1}$ | Risco de divergencia | Raramente |

### Learning Rate Scheduling
Na pratica, $\eta$ nao precisa ser constante:
- **Step decay:** reduzir $\eta$ a cada N epocas
- **Cosine annealing:** $\eta_t = \eta_0 \cdot \frac{1 + \cos(\pi t / T)}{2}$
- **Warmup:** comecar pequeno e aumentar nas primeiras epocas

**Por que em ML:** Learning rate eh o hiperparametro que mais afeta o treinamento. Um $\eta$ errado pode fazer a diferenca entre um modelo que converge em minutos e um que nunca converge.

In [5]:
# Comparar diferentes learning rates
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

learning_rates = [0.01, 0.1, 0.5, 1.5]

for idx, eta in enumerate(learning_rates):
    ax = axes[idx // 2, idx % 2]
    
    x = 5.0
    history = [x]
    losses = [f(x)]
    
    for _ in range(30):
        if x > 100 or x < -100:
            break
        grad = df(x)
        x = x - eta * grad
        history.append(x)
        losses.append(f(x))
    
    iterations = np.arange(len(losses))
    ax.plot(iterations, losses, linewidth=2, marker='o', label=f'eta={eta}')
    ax.set_xlabel('Iteracao')
    ax.set_ylabel('Perda')
    ax.set_title(f'Learning Rate = {eta}')
    ax.set_yscale('log')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, which='both')
    
    if len(losses) < 30:
        ax.text(0.5, 0.5, f'DIVERGIU!', transform=ax.transAxes, fontsize=14, 
                color='red', ha='center', va='center', 
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.suptitle('Importancia do Learning Rate')
plt.tight_layout()
plt.show()

print('Observacoes:')
print('eta=0.01: convergencia lenta')
print('eta=0.1: convergencia balanceada')
print('eta=0.5: ainda funciona')
print('eta=1.5: DIVERGE!')

Observacoes:
eta=0.01: convergencia lenta
eta=0.1: convergencia balanceada
eta=0.5: ainda funciona
eta=1.5: DIVERGE!


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/1878159126.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- $\eta = 0.01$: convergencia lenta mas segura (muitas iteracoes ate chegar perto do minimo)
- $\eta = 0.1$: convergencia rapida e suave (bom equilibrio)
- $\eta = 0.5$: ainda converge, mas com oscilacoes visiveis
- $\eta = 1.5$: DIVERGE completamente (a loss explode!)

**O que concluir:**
- Existe um intervalo "seguro" para $\eta$, e ele depende da curvatura da funcao
- Formalmente, para funcoes convexas com constante de Lipschitz $L$: $\eta < 2/L$ garante convergencia
- Na pratica: comece com $\eta = 10^{-3}$ e ajuste. Se loss oscila, reduza. Se converge muito devagar, aumente
- Learning rate scheduling (reduzir $\eta$ ao longo do treinamento) combina o melhor dos dois mundos

**Conexao com 0.4:** A condicao de convergencia $\eta < 2/L$ envolve a segunda derivada (curvatura). Quanto mais "curva" a funcao (segunda derivada grande), menor precisa ser o passo.

## 6. Batch GD vs SGD vs Mini-batch

### Analogia: Pesquisa de Opiniao

Voce quer saber a opiniao media de uma cidade (= gradiente verdadeiro):
- **Batch GD:** perguntar para TAREFA DO ALUNOS os habitantes. Resultado preciso, mas leva meses
- **SGD:** perguntar para 1 pessoa aleatoria. Rapido, mas ruidoso (pode ser um outlier)
- **Mini-batch:** perguntar para um grupo de 32 pessoas. Rapido E razoavelmente preciso

### Formalizacao

| Metodo | Gradiente estimado | Custo/iteracao | Ruido |
|--------|-------------------|----------------|-------|
| Batch GD | $\nabla\mathcal{L} = \frac{1}{n}\sum_{i=1}^n \nabla\ell_i$ | $O(n)$ | Zero |
| SGD | $\nabla\ell_j$ (1 amostra) | $O(1)$ | Alto |
| Mini-batch | $\frac{1}{k}\sum_{j \in B} \nabla\ell_j$ ($k$ amostras) | $O(k)$ | Medio |

**Por que em ML:** Datasets modernos tem milhoes de amostras. Batch GD eh proibitivo. SGD puro eh muito ruidoso. Mini-batch (tipicamente $k = 32$ a $256$) eh o padrao universal.

**Conexao com 0.6:** O ruido do SGD eh um fenomeno estatistico: estamos estimando o gradiente medio a partir de uma amostra. A variancia da estimativa diminui com $\sqrt{k}$ (Lei dos Grandes Numeros, 0.6).

In [6]:
# Simular otimizacao com diferentes metodos
np.random.seed(42)
n_samples = 100
X = np.random.randn(n_samples, 2)
w_true = np.array([3, -2])
y = X @ w_true + np.random.randn(n_samples) * 0.1

def mse_loss(w, X, y):
    return np.mean((y - X @ w)**2)

def gradient(w, X, y):
    residuals = y - X @ w
    return -2 * X.T @ residuals / len(y)

eta = 0.01
iterations = 200
w0 = np.array([0.0, 0.0])

# Batch GD
w_batch = w0.copy()
hist_batch = [mse_loss(w_batch, X, y)]
for _ in range(iterations):
    w_batch = w_batch - eta * gradient(w_batch, X, y)
    hist_batch.append(mse_loss(w_batch, X, y))

# SGD
w_sgd = w0.copy()
hist_sgd = [mse_loss(w_sgd, X, y)]
for _ in range(iterations):
    idx = np.random.randint(0, n_samples)
    grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w_sgd) / 1
    w_sgd = w_sgd - eta * grad
    hist_sgd.append(mse_loss(w_sgd, X, y))

# Mini-batch
w_mini = w0.copy()
hist_mini = [mse_loss(w_mini, X, y)]
batch_size = 16
for _ in range(iterations):
    idx = np.random.choice(n_samples, batch_size, replace=False)
    grad = gradient(w_mini, X[idx], y[idx])
    w_mini = w_mini - eta * grad
    hist_mini.append(mse_loss(w_mini, X, y))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hist_batch, linewidth=2, label='Batch GD (suave)', color='blue')
ax.plot(hist_sgd, linewidth=1, alpha=0.7, label='SGD (ruidoso)', color='red')
ax.plot(hist_mini, linewidth=2, label='Mini-batch (equilibrio)', color='green')
ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Batch vs SGD vs Mini-batch')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Caracteristicas:')
print('Batch GD: suave, deterministico, lento')
print('SGD: ruidoso, rapido, pode escapar minimos locais')
print('Mini-batch: equilibrio')

Caracteristicas:
Batch GD: suave, deterministico, lento
SGD: ruidoso, rapido, pode escapar minimos locais
Mini-batch: equilibrio


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/458347728.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Batch GD (azul): curva perfeitamente suave, convergencia monotona
- SGD (vermelho): curva muito ruidosa, oscila bastante mesmo perto do minimo
- Mini-batch (verde): meio-termo; suave o suficiente mas com algum ruido saudavel

**O que concluir:**
- O ruido do SGD nao eh so desvantagem: em problemas nao-convexos, o ruido ajuda a **escapar de minimos locais ruins**
- Mini-batch eh o padrao na pratica porque: (a) rapido por iteracao, (b) suave o suficiente, (c) paralelizavel em GPU
- Batch size eh hiperparametro: muito grande = pouco ruido (pode ficar preso em minimo local), muito pequeno = muito ruido

**Conexao com 0.7 (LGN):** A media de mini-batch converge para o gradiente verdadeiro conforme $k \to \infty$. Com $k$ finito, temos um estimador nao-viesado com variancia $\sigma^2/k$.

## 7. SGD com Momentum

### Analogia: Bola de Boliche na Rampa

SGD puro eh como uma bolinha de ping-pong descendo uma rampa: qualquer brisa muda sua direcao. Momentum transforma isso numa **bola de boliche**: ela acumula velocidade e nao muda de direcao facilmente.

### Algoritmo

$$v_t = \beta \cdot v_{t-1} + (1 - \beta) \cdot \nabla\mathcal{L}(w_t)$$
$$w_{t+1} = w_t - \eta \cdot v_t$$

onde $\beta \in [0, 1)$ controla quanto do "momento" passado manter. Tipicamente $\beta = 0.9$.

### Intuicao Matematica
$v_t$ eh uma **media movel exponencial** dos gradientes. Se os ultimos gradientes apontam consistentemente na mesma direcao, $v_t$ cresce nessa direcao (aceleracao). Se oscilam, $v_t$ fica pequeno (amortecimento).

**Por que em ML:** Momentum resolve dois problemas: (1) acelera convergencia em direcoes consistentes, (2) reduz oscilacoes em direcoes ruidosas. Eh o primeiro passo de SGD para otimizadores modernos.

**Conexao com 0.5:** A media movel exponencial eh $v_t = (1-\beta)\sum_{k=0}^{t} \beta^k g_{t-k}$, uma serie geometrica ponderada dos gradientes passados.

In [7]:
# Implementar Momentum
def sgd_momentum(X, y, eta=0.01, beta=0.9, iterations=200):
    w = np.array([0.0, 0.0])
    v = np.array([0.0, 0.0])
    losses = [mse_loss(w, X, y)]
    
    for _ in range(iterations):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        v = beta * v + (1 - beta) * grad
        w = w - eta * v
        losses.append(mse_loss(w, X, y))
    
    return losses

hist_momentum = sgd_momentum(X, y, beta=0.9)
hist_no_momentum = sgd_momentum(X, y, beta=0.0)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hist_no_momentum, linewidth=1, alpha=0.7, label='SGD sem momentum', color='red')
ax.plot(hist_momentum, linewidth=2, label='SGD com momentum (beta=0.9)', color='green')
ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Momentum reduz oscilacoes')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Momentum:')
print('- Acumula velocidade na direcao certa')
print('- Reduz oscilacoes')
print('- Acelera convergencia')

Momentum:
- Acumula velocidade na direcao certa
- Reduz oscilacoes
- Acelera convergencia


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/3183793829.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Sem momentum (vermelho): curva oscila bastante, convergencia irregular
- Com momentum $\beta = 0.9$ (verde): curva muito mais suave, convergencia mais rapida e estavel

**O que concluir:**
- Momentum eh uma melhoria simples mas poderosa sobre SGD puro
- $\beta = 0.9$ eh valor padrao: carrega 90% da velocidade anterior
- O custo computacional eh minimo: apenas uma multiplicacao e soma extras por iteracao
- Momentum eh ingrediente fundamental do Adam (secao 10)

**Conexao com 0.5:** O fator $\beta^k$ decai exponencialmente, entao gradientes recentes pesam mais. A "memoria efetiva" eh $\sim 1/(1-\beta) = 10$ iteracoes para $\beta = 0.9$.

## 8. AdaGrad: Learning Rate Adaptativo por Parametro

### Analogia: Professor que Ajusta a Dificuldade

Imagine um professor que ajusta a dificuldade para cada aluno individualmente:
- Alunos que ja erraram muito (gradientes grandes acumulados) recebem exercicios mais faceis (learning rate menor)
- Alunos que mal foram testados (gradientes pequenos acumulados) recebem exercicios normais (learning rate maior)

### Algoritmo

$$s_t = s_{t-1} + (\nabla\mathcal{L})^2 \qquad \text{(acumula quadrado dos gradientes)}$$
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{s_t + \epsilon}} \cdot \nabla\mathcal{L}$$

Cada parametro $w_j$ tem seu proprio $s_j$, portanto seu proprio learning rate efetivo.

**Por que em ML:** Util quando features tem escalas muito diferentes (NLP: palavras frequentes vs raras). Parametros de features raras recebem updates maiores, frequentes recebem menores.

**Problema:** $s$ so cresce (soma de quadrados). Com o tempo, o learning rate efetivo $\eta/\sqrt{s}$ vai a zero e o modelo para de aprender. RMSprop resolve isso.

In [8]:
# Implementar AdaGrad
def adagrad(X, y, eta=0.1, iterations=200, epsilon=1e-8):
    w = np.array([0.0, 0.0])
    s = np.array([0.0, 0.0])
    losses = [mse_loss(w, X, y)]
    
    for _ in range(iterations):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        grad = grad.ravel()
        s = s + grad**2
        w = w - eta * grad / (np.sqrt(s) + epsilon)
        losses.append(mse_loss(w, X, y))
    
    return losses

hist_adagrad = adagrad(X, y, eta=0.1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hist_batch, linewidth=2, alpha=0.7, label='Batch GD', color='blue')
ax.plot(hist_sgd, linewidth=1, alpha=0.5, label='SGD', color='red')
ax.plot(hist_adagrad, linewidth=2, label='AdaGrad', color='purple')
ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Otimizadores: Batch GD vs SGD vs AdaGrad')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('AdaGrad:')
print('- Parametros frequentes: learning rate menor')
print('- Parametros raros: learning rate maior')
print('- Problema: learning rate pode ficar muito pequeno')

AdaGrad:
- Parametros frequentes: learning rate menor
- Parametros raros: learning rate maior
- Problema: learning rate pode ficar muito pequeno


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/2376758099.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- AdaGrad (roxo) converge mais rapido que SGD (vermelho) nas primeiras iteracoes
- Comparado com Batch GD (azul), AdaGrad adapta automaticamente sem precisar tunar $\eta$ cuidadosamente

**O que concluir:**
- Learning rates adaptativos sao mais robustos que learning rate fixo
- AdaGrad funciona bem para problemas com features esparsas (NLP, recomendacao)
- O problema fundamental: $s$ cresce monotonicamente, entao o learning rate so diminui. Em treinamentos longos, o modelo "congela"
- Isso motivou o RMSprop (proxima secao): usar decay ao inves de soma infinita

**Conexao com 0.4:** O denominador $\sqrt{s + \epsilon}$ age como uma estimativa da curvatura (segunda derivada) de cada parametro. Parametros com curvatura alta (gradientes grandes) recebem passos menores -- similar ao metodo de Newton, mas sem calcular a Hessiana.

## 9. RMSprop: Decaimento Exponencial do Historico

### Analogia: Memoria Seletiva

AdaGrad lembra TUDO (inclusive gradientes de meses atras). RMSprop tem "memoria seletiva": lembra fortemente o passado recente e esquece gradualmente o passado distante.

### Algoritmo

$$s_t = \rho \cdot s_{t-1} + (1 - \rho) \cdot (\nabla\mathcal{L})^2$$
$$w_{t+1} = w_t - \frac{\eta}{\sqrt{s_t + \epsilon}} \cdot \nabla\mathcal{L}$$

A unica diferenca do AdaGrad: usar **media movel exponencial** ($\rho = 0.9$) ao inves de soma cumulativa.

**Por que em ML:** RMSprop funciona melhor que AdaGrad em treinamentos longos (redes neurais treinadas por muitas epocas), porque o learning rate efetivo nao vai a zero.

**Conexao com 0.5:** A media movel exponencial $s_t = (1-\rho)\sum_{k=0}^{t} \rho^k g_{t-k}^2$ eh novamente uma serie geometrica. A "janela efetiva" eh $\sim 1/(1-\rho) = 10$ iteracoes para $\rho = 0.9$.

In [9]:
# Implementar RMSprop
def rmsprop(X, y, eta=0.01, rho=0.9, iterations=200, epsilon=1e-8):
    w = np.array([0.0, 0.0])
    s = np.array([0.0, 0.0])
    losses = [mse_loss(w, X, y)]
    
    for _ in range(iterations):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        grad = grad.ravel()
        s = rho * s + (1 - rho) * grad**2
        w = w - eta * grad / (np.sqrt(s) + epsilon)
        losses.append(mse_loss(w, X, y))
    
    return losses

hist_rmsprop = rmsprop(X, y, eta=0.01)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hist_batch, linewidth=2, alpha=0.7, label='Batch GD', color='blue')
ax.plot(hist_adagrad, linewidth=2, alpha=0.7, label='AdaGrad', color='purple')
ax.plot(hist_rmsprop, linewidth=2, label='RMSprop', color='orange')
ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('RMSprop: Decay exponencial')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('RMSprop:')
print('- Melhora AdaGrad com decay')
print('- Usa media movel ao inves de soma')
print('- Melhor comportamento em longo prazo')

RMSprop:
- Melhora AdaGrad com decay
- Usa media movel ao inves de soma
- Melhor comportamento em longo prazo


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/651025622.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- RMSprop (laranja) mantem convergencia estavel mesmo em iteracoes tardias
- Comparado com AdaGrad (roxo), o learning rate efetivo nao decai para zero

**O que concluir:**
- O decaimento exponencial resolve o problema fundamental do AdaGrad
- $\rho = 0.9$ eh valor padrao (equivale a janela de ~10 iteracoes)
- RMSprop foi proposto por Hinton numa aula (nao num paper!), mostrando que boas ideias podem surgir informalmente
- RMSprop + Momentum = Adam (proxima secao)

**Conexao com 0.7:** A media movel exponencial eh essencialmente um estimador bayesiano com prior exponencial no passado recente.

## 10. Adam: Adaptive Moment Estimation

### Analogia: GPS com Velocidade Adaptativa

Adam eh como um GPS que:
1. **Lembra a direcao recente** (Momentum, 1o momento): se voce vinha indo para o norte, continua acelerando para o norte
2. **Ajusta a velocidade por terreno** (RMSprop, 2o momento): em terreno plano anda rapido, em terreno ingreme anda devagar

### Algoritmo Completo

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla\mathcal{L} \qquad \text{(1o momento: media dos gradientes)}$$
$$s_t = \beta_2 s_{t-1} + (1 - \beta_2) (\nabla\mathcal{L})^2 \qquad \text{(2o momento: variancia dos gradientes)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{s}_t = \frac{s_t}{1 - \beta_2^t} \qquad \text{(correcao de vies)}$$
$$w_{t+1} = w_t - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{s}_t} + \epsilon}$$

### Hiperparametros Padrao
$\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.001$, $\epsilon = 10^{-8}$

**Por que em ML:** Adam eh o otimizador padrao em deep learning. Se voce nao sabe qual otimizador usar, use Adam. Funciona bem em ~90% dos casos sem tuning.

**Conexao com secoes anteriores:** Adam = Momentum (secao 7) + RMSprop (secao 9) + correcao de vies para as primeiras iteracoes.

In [10]:
# Implementar Adam
def adam(X, y, eta=0.001, beta1=0.9, beta2=0.999, iterations=200, epsilon=1e-8):
    w = np.array([0.0, 0.0])
    m = np.array([0.0, 0.0])
    s = np.array([0.0, 0.0])
    losses = [mse_loss(w, X, y)]
    
    for t in range(1, iterations + 1):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        grad = grad.ravel()
        
        m = beta1 * m + (1 - beta1) * grad
        s = beta2 * s + (1 - beta2) * grad**2
        
        m_hat = m / (1 - beta1**t)
        s_hat = s / (1 - beta2**t)
        
        w = w - eta * m_hat / (np.sqrt(s_hat) + epsilon)
        losses.append(mse_loss(w, X, y))
    
    return losses

hist_adam = adam(X, y, eta=0.01)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hist_batch, linewidth=2, alpha=0.6, label='Batch GD', color='blue')
ax.plot(hist_momentum, linewidth=2, alpha=0.6, label='Momentum', color='green')
ax.plot(hist_rmsprop, linewidth=2, alpha=0.6, label='RMSprop', color='orange')
ax.plot(hist_adam, linewidth=2, label='Adam', color='red')
ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_yscale('log')
ax.set_title('Comparacao de Otimizadores')
ax.legend(fontsize=11)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print('Adam:')
print('- Combina melhores ideias')
print('- Mais robusto que alternativas')
print('- Default em deep learning')
print('- Pratico sempre funciona')

Adam:
- Combina melhores ideias
- Mais robusto que alternativas
- Default em deep learning
- Pratico sempre funciona


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/216127784.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Adam (vermelho) converge mais rapido e mais suave que todos os outros otimizadores
- A combinacao de momentum (suavidade) + learning rate adaptativo (robustez) eh superior a qualquer um sozinho
- Ate o Batch GD (azul), que usa o gradiente exato, eh mais lento que Adam com gradientes estocasticos

**O que concluir:**
- Adam eh o "canivete suico" dos otimizadores: funciona bem em quase qualquer situacao
- A correcao de vies ($\hat{m}_t, \hat{s}_t$) eh crucial: sem ela, as primeiras iteracoes sao viesadas para zero
- Na pratica, os defaults ($\beta_1=0.9$, $\beta_2=0.999$, $\eta=0.001$) funcionam surpreendentemente bem
- Excecoes: para treinamento de GANs ou tarefas muito especificas, SGD com momentum pode superar Adam

**Conexao com 0.7:** A correcao de vies compensa o fato de $m_0 = 0$ e $s_0 = 0$. Sem correcao, as primeiras estimativas sao viesadas para zero. Isso eh analogo a correcao de Bessel ($n-1$) na estimativa de variancia amostral.

## 11. Regularizacao (L1 e L2)

### Analogia: Navalha de Occam para Modelos

Regularizacao implementa o principio "modelos simples sao melhores":
- **L2 (Ridge):** "todos os parametros devem ser pequenos" $\to$ penaliza $\lambda\|w\|_2^2$
- **L1 (Lasso):** "muitos parametros devem ser zero" $\to$ penaliza $\lambda\|w\|_1$

### Formulacao

$$\mathcal{L}_{\text{reg}}(w) = \mathcal{L}(w) + \lambda \cdot R(w)$$

| Tipo | $R(w)$ | Efeito | Quando usar |
|------|--------|--------|-------------|
| L2 (Ridge) | $\sum w_j^2$ | Pesos pequenos, nunca exatamente zero | Sempre (default) |
| L1 (Lasso) | $\sum |w_j|$ | Muitos pesos exatamente zero (sparsidade) | Selecao de features |
| Elastic Net | $\alpha\|w\|_1 + (1-\alpha)\|w\|_2^2$ | Combinacao | Muitas features correlacionadas |

### Perspectiva Bayesiana (conexao com 0.7)
- L2 = prior gaussiano: $w \sim \mathcal{N}(0, 1/\lambda)$
- L1 = prior Laplaciano: $w \sim \text{Laplace}(0, 1/\lambda)$

O prior Laplaciano tem "pico" em zero, incentivando sparsidade.

**Por que em ML:** Regularizacao eh a principal defesa contra overfitting. Em redes neurais, L2 (weight decay) eh usado por padrao. $\lambda$ controla o trade-off entre ajustar os dados (baixo bias) e manter o modelo simples (baixa variancia).

In [11]:
# Visualizar efeito de regularizacao
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

lambdas = [0.0, 0.1, 1.0]

for idx, lam in enumerate(lambdas):
    ax = axes[idx]
    
    w_range = np.linspace(-3, 3, 100)
    mse_values = (w_range - 2)**2
    l2_penalty = lam * w_range**2
    total = mse_values + l2_penalty
    
    ax.plot(w_range, mse_values, label='MSE original', linewidth=2, color='blue')
    ax.plot(w_range, l2_penalty, label=f'L2 penalty (lambda={lam})', linewidth=2, color='red')
    ax.plot(w_range, total, label='Total (MSE + L2)', linewidth=3, color='green')
    
    min_idx = np.argmin(total)
    ax.scatter([w_range[min_idx]], [total[min_idx]], s=200, color='green', marker='*')
    
    ax.set_xlabel('w')
    ax.set_ylabel('Custo')
    ax.set_title(f'Regularizacao L2: lambda={lam}')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Regularizacao:')
print('- lambda=0: sem regularizacao')
print('- lambda grande: pesos penalizados')
print('- Evita overfitting')

Regularizacao:
- lambda=0: sem regularizacao
- lambda grande: pesos penalizados
- Evita overfitting


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/1922690726.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- $\lambda = 0$ (sem regularizacao): minimo em $w = 2$ (valor verdadeiro)
- $\lambda = 0.1$: minimo deslocado levemente em direcao a zero (penalty leve)
- $\lambda = 1.0$: minimo deslocado significativamente para zero (penalty forte)
- A estrela verde mostra o minimo total (MSE + penalidade)

**O que concluir:**
- Regularizacao "puxa" os parametros em direcao a zero, combatendo overfitting
- $\lambda$ grande demais causa underfitting (modelo muito simples, ignora os dados)
- $\lambda$ pequeno demais nao regulariza o suficiente (overfitting persiste)
- O valor otimo de $\lambda$ eh encontrado por cross-validation

**Conexao com 0.7 (Bayesiana):** Regularizacao L2 = MAP estimation com prior gaussiano sobre $w$. A forca $\lambda$ corresponde a precisao ($1/\sigma^2$) do prior. $\lambda$ grande = prior forte = mais confianca de que $w \approx 0$.

## 12. Vanishing e Exploding Gradients

### Analogia: Telefone sem Fio

Numa rede neural profunda com $L$ camadas, o gradiente precisa "viajar" da ultima camada ate a primeira (backpropagation). Eh como um telefone sem fio:
- Se cada pessoa sussurra mais baixo que a anterior (fator $< 1$): a mensagem desaparece (**vanishing**)
- Se cada pessoa grita mais alto ($> 1$): vira caos (**exploding**)

### Matematica

Pela regra da cadeia (0.4):

$$\frac{\partial\mathcal{L}}{\partial w_1} = \frac{\partial\mathcal{L}}{\partial h_L} \cdot \frac{\partial h_L}{\partial h_{L-1}} \cdots \frac{\partial h_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial w_1}$$

Se $\left|\frac{\partial h_k}{\partial h_{k-1}}\right| < 1$ para todos os $k$: gradiente decai como $r^L \to 0$ (vanishing)

Se $\left|\frac{\partial h_k}{\partial h_{k-1}}\right| > 1$ para todos os $k$: gradiente cresce como $r^L \to \infty$ (exploding)

### Solucoes Modernas
| Problema | Solucao | Como funciona |
|----------|---------|---------------|
| Vanishing (sigmoid/tanh) | ReLU | Derivada = 1 para $x > 0$ (nao encolhe) |
| Vanishing/Exploding | Batch Normalization | Normaliza ativacoes em cada camada |
| Vanishing em redes muito profundas | Skip Connections (ResNet) | Gradiente "atalha" por conexoes diretas |
| Exploding | Gradient Clipping | Limita norma do gradiente a um maximo |

**Por que em ML:** Vanishing/exploding gradients sao o principal obstculo para treinar redes profundas. As solucoes acima (ReLU, BatchNorm, ResNet) sao por que deep learning funciona na pratica.

**Conexao com 0.4:** Tudo vem da regra da cadeia! O produto de muitas derivadas parciais pode crescer ou encolher exponencialmente.

In [12]:
# Simular vanishing gradients em rede profunda
n_layers = 30
weights = np.ones(n_layers)

# Cenario 1: gradientes diminuem
grad_vanish = np.ones(n_layers)
for i in range(n_layers - 1):
    grad_vanish[i+1] = grad_vanish[i] * 0.95

# Cenario 2: gradientes crescem
grad_explode = np.ones(n_layers)
for i in range(n_layers - 1):
    grad_explode[i+1] = grad_explode[i] * 1.1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(grad_vanish, linewidth=2, marker='o', markersize=4, color='blue')
ax1.set_xlabel('Camada')
ax1.set_ylabel('Magnitude do gradiente')
ax1.set_yscale('log')
ax1.set_title('Vanishing Gradient: diminui exponencialmente')
ax1.grid(alpha=0.3, which='both')

ax2.plot(grad_explode, linewidth=2, marker='o', markersize=4, color='red')
ax2.set_xlabel('Camada')
ax2.set_ylabel('Magnitude do gradiente')
ax2.set_yscale('log')
ax2.set_title('Exploding Gradient: cresce exponencialmente')
ax2.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print('Problema em redes profundas:')
print('- Backprop multiplica gradientes')
print('- Se |peso| < 1: vanishing')
print('- Se |peso| > 1: exploding')
print('- Solucoes: ReLU, batch norm, skip connections')

Problema em redes profundas:
- Backprop multiplica gradientes
- Se |peso| < 1: vanishing
- Se |peso| > 1: exploding
- Solucoes: ReLU, batch norm, skip connections


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/3543297561.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Vanishing (esquerdo): gradiente decai exponencialmente, camadas iniciais recebem gradiente ~0 e nao aprendem
- Exploding (direito): gradiente cresce exponencialmente, atualizacoes ficam gigantescas e o modelo diverge
- Note a escala logaritmica: o decaimento/crescimento eh exponencial na profundidade

**O que concluir:**
- Para 30 camadas com fator 0.95: gradiente cai para $0.95^{30} \approx 0.21$ (21% do original)
- Para 30 camadas com fator 1.1: gradiente cresce para $1.1^{30} \approx 17.4$ (1740% do original!)
- Redes muito profundas (100+ camadas) so sao possiveis gracas a skip connections (ResNet)
- Este eh o motivo pelo qual sigmoid/tanh foram substituidos por ReLU como ativacao padrao

**Conexao com 0.5 (Series):** O gradiente total eh um produto de $L$ termos. Se cada termo eh $r$, o produto eh $r^L$ -- uma progressao geometrica cuja convergencia/divergencia depende de $|r| \lessgtr 1$.

## 13. Aplicacao: Regressao Linear do Zero

### Por que implementar do zero?

Implementar regressao linear com GD junta TUDO que aprendemos:
- Funcao de custo: MSE (secao 2)
- Otimizacao: Gradient Descent (secao 4)
- Gradiente: derivada parcial do MSE em relacao a $w$ e $b$ (0.4)
- Convergencia: problema convexo, garantia de encontrar o minimo (secao 3)

### Modelo

$$\hat{y} = w_1 x + w_0$$

### Gradientes

$$\frac{\partial \text{MSE}}{\partial w} = -\frac{2}{n} X^T(y - \hat{y})$$

**Por que em ML:** Regressao linear eh o modelo mais simples de ML. Se voce entende como treina-lo com GD, entende o principio por tras de QUALQUER modelo mais complexo (logistica, redes neurais, etc.).

In [13]:
# Dataset simples
X_train = np.linspace(0, 10, 50).reshape(-1, 1)
w_true = 2.5
b_true = 1.0
y_train = w_true * X_train.ravel() + b_true + np.random.randn(50) * 0.5

# Adicionar coluna de 1s para bias
X_train_augmented = np.column_stack([np.ones(len(X_train)), X_train])

# GD
w = np.array([0.0, 0.0])
eta = 0.01
losses = []

for _ in range(100):
    y_pred = X_train_augmented @ w
    mse = np.mean((y_train - y_pred)**2)
    losses.append(mse)
    
    grad = -2 * X_train_augmented.T @ (y_train - y_pred) / len(y_train)
    w = w - eta * grad

w_learned = w

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Convergencia
ax1.plot(losses, linewidth=2, color='red')
ax1.set_xlabel('Iteracao')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Convergencia: MSE diminui')
ax1.grid(alpha=0.3)

# Fit
ax2.scatter(X_train, y_train, alpha=0.6, s=50, label='Dados')
X_range = np.linspace(X_train.min(), X_train.max(), 100)
y_pred_range = w_learned[0] + w_learned[1] * X_range
ax2.plot(X_range, y_pred_range, 'r-', linewidth=2, label='Regressao aprendida')
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Regressao Linear')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Parametros verdadeiros: w={w_true:.3f}, b={b_true:.3f}')
print(f'Parametros aprendidos: w={w_learned[1]:.3f}, b={w_learned[0]:.3f}')
print(f'Erro final MSE: {losses[-1]:.4f}')

Parametros verdadeiros: w=2.500, b=1.000
Parametros aprendidos: w=2.557, b=0.633
Erro final MSE: 0.3337


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/3347155054.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Grafico de convergencia (esquerdo): MSE cai rapidamente e estabiliza (problema convexo!)
- Grafico de fit (direito): a reta aprendida (vermelha) aproxima bem os dados
- Os parametros aprendidos devem estar proximos dos verdadeiros ($w \approx 2.5$, $b \approx 1.0$)

**O que concluir:**
- Regressao linear + MSE + GD funciona perfeitamente: convergencia garantida por convexidade
- O mesmo framework (definir custo, calcular gradiente, iterar) se aplica a QUALQUER modelo
- Com 100 iteracoes e $\eta = 0.01$, ja chegamos muito perto da solucao exata (que poderiamos obter por $w^* = (X^TX)^{-1}X^Ty$)
- A solucao analitica eh mais rapida aqui, mas GD escala para milhoes de amostras e modelos nao-lineares

**Conexao com 0.3:** A solucao analitica $w^* = (X^TX)^{-1}X^Ty$ usa inversao de matriz (algebra linear). GD eh uma alternativa que nao precisa inverter matrizes -- crucial quando $X^TX$ eh muito grande.

## 14. Aplicacao: Regressao Logistica do Zero

### Da Regressao a Classificacao

Regressao logistica adapta o framework de otimizacao para **classificacao binaria**:
1. Modelo: $\hat{y} = \sigma(w^T x + b)$ onde $\sigma(z) = \frac{1}{1 + e^{-z}}$ (sigmoid)
2. Custo: Cross-Entropy (nao MSE! -- ver secao 2)
3. Otimizacao: Gradient Descent (mesmo algoritmo!)

### Por que Cross-Entropy?

MSE para classificacao tem gradientes que "encolhem" perto de 0 e 1 (sigmoid satura), causando vanishing gradients. Cross-entropy tem gradientes proporcionais ao erro, convergindo mais rapido.

**Conexao com 0.7 (MLE):** Minimizar cross-entropy = maximizar likelihood de Bernoulli. A sigmoid transforma o logit linear em probabilidade $P(y=1|x)$.

**Por que em ML:** Regressao logistica eh a base de redes neurais. Cada neuronio eh essencialmente uma regressao logistica com ativacao nao-linear.

In [14]:
# Dataset simples
np.random.seed(42)
X_class0 = np.random.randn(50, 2) - 1
X_class1 = np.random.randn(50, 2) + 1
X_logistic = np.vstack([X_class0, X_class1])
y_logistic = np.hstack([np.zeros(50), np.ones(50)])

X_logistic_aug = np.column_stack([np.ones(len(X_logistic)), X_logistic])

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# GD para logistica
w_log = np.array([0.0, 0.0, 0.0])
eta = 0.1
losses_log = []

for _ in range(100):
    logits = X_logistic_aug @ w_log
    y_pred_prob = sigmoid(logits)
    
    ce_loss = -np.mean(y_logistic * np.log(y_pred_prob + 1e-10) + 
                        (1 - y_logistic) * np.log(1 - y_pred_prob + 1e-10))
    losses_log.append(ce_loss)
    
    grad = X_logistic_aug.T @ (y_pred_prob - y_logistic) / len(y_logistic)
    w_log = w_log - eta * grad

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses_log, linewidth=2, color='red')
ax1.set_xlabel('Iteracao')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Convergencia: Loss diminui')
ax1.grid(alpha=0.3)

ax2.scatter(X_class0[:, 0], X_class0[:, 1], label='Classe 0', alpha=0.6, s=50)
ax2.scatter(X_class1[:, 0], X_class1[:, 1], label='Classe 1', alpha=0.6, s=50)

x_range = np.linspace(X_logistic[:, 0].min() - 1, X_logistic[:, 0].max() + 1, 100)
y_range = np.linspace(X_logistic[:, 1].min() - 1, X_logistic[:, 1].max() + 1, 100)
xx, yy = np.meshgrid(x_range, y_range)
Z = sigmoid(w_log[0] + w_log[1] * xx + w_log[2] * yy)
ax2.contourf(xx, yy, Z, levels=20, alpha=0.3, cmap='RdYlBu')
ax2.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)

ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Fronteira de Decisao')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Loss final: {losses_log[-1]:.4f}')
print('Regressao logistica convergiu!')

Loss final: 0.1687
Regressao logistica convergiu!


/var/folders/pg/j2d1lxdx6yq8d2lg7n2l5npw0000gn/T/ipykernel_39372/1139724098.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**O que observar:**
- Grafico de convergencia (esquerdo): cross-entropy cai consistentemente
- Grafico de fronteira (direito): a reta de decisao (preta) separa as duas classes
- O contour mostra probabilidades: azul = classe 0, vermelho = classe 1, com gradiente suave entre elas

**O que concluir:**
- O mesmo framework (custo + gradiente + iteracao) funciona para classificacao -- so muda o custo e o modelo
- A fronteira de decisao eh linear ($w^Tx + b = 0$), mas as probabilidades variam suavemente via sigmoid
- Regressao logistica eh convexa: nao importa a inicializacao, GD encontra o minimo global
- Na pratica, usariamos regularizacao (secao 11) para evitar overfitting em problemas reais

**Conexao com 0.7:** O output $\sigma(w^Tx + b)$ eh literalmente $P(y=1|x)$ sob o modelo. A fronteira de decisao esta onde $P(y=1|x) = 0.5$, ou seja, $w^Tx + b = 0$.

## 15. Exercicios Praticos

### Exercicio 1: Learning Rate Grid Search
Implemente uma funcao que testa diferentes learning rates para GD numa parabola $f(x) = (x-3)^2$ e retorna o melhor $\eta$. Defina "melhor" como aquele que atinge loss $< 0.01$ no menor numero de iteracoes.

**Dica:** teste $\eta \in \{0.001, 0.01, 0.05, 0.1, 0.3, 0.5\}$, comece com $x_0 = 8$, rode ate 100 iteracoes.

In [15]:
# EXERCICIO 1: Learning Rate Grid Search
# Complete a funcao abaixo

def lr_grid_search(etas, x0=8.0, max_iter=100, target=0.01):
    """
    Testa diferentes learning rates para minimizar f(x) = (x-3)^2.

    Retorna:
        best_eta: learning rate que atingiu target em menos iteracoes
        results: dict {eta: n_iteracoes_para_target (ou max_iter se nao convergiu)}
    """
    def f(x): return (x - 3)**2
    def df(x): return 2*(x - 3)

    results = {}

    for eta in etas:
        x = x0
        for i in range(max_iter):
            if f(x) < target:
                results[eta] = i
                break
            x = None  # TAREFA DO ALUNO: atualizar x usando GD
        else:
            results[eta] = max_iter  # nao convergiu

    best_eta = None  # TAREFA DO ALUNO: encontrar eta com menor numero de iteracoes
    return best_eta, results

# Teste
etas = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
# best, results = lr_grid_search(etas)
# print(f"Melhor eta: {best}")
# for eta, iters in results.items():
#     print(f"  eta={eta}: {iters} iteracoes")

In [16]:
# SOLUCAO Exercicio 1

def lr_grid_search(etas, x0=8.0, max_iter=100, target=0.01):
    def f(x): return (x - 3)**2
    def df(x): return 2*(x - 3)

    results = {}

    for eta in etas:
        x = x0
        converged = False
        for i in range(max_iter):
            if f(x) < target:
                results[eta] = i
                converged = True
                break
            x = x - eta * df(x)
        if not converged:
            # Verificar divergencia
            if abs(x) > 1e6:
                results[eta] = -1  # divergiu
            else:
                results[eta] = max_iter

    # Melhor eta: menor iteracoes positivas
    valid = {k: v for k, v in results.items() if v > 0}
    best_eta = min(valid, key=valid.get) if valid else None
    return best_eta, results

etas = [0.001, 0.01, 0.05, 0.1, 0.3, 0.5]
best, results = lr_grid_search(etas)

print(f"Melhor learning rate: {best}")
print(f"{'eta':>8} | {'Iteracoes':>10} | {'Status':>10}")
print("-" * 35)
for eta, iters in sorted(results.items()):
    status = "DIVERGIU" if iters == -1 else ("NAO CONV" if iters == 100 else "OK")
    print(f"{eta:>8.3f} | {iters:>10} | {status:>10}")

# Visualizar
fig, ax = plt.subplots(figsize=(10, 5))
for eta in [0.01, 0.1, 0.3]:
    x = 8.0
    losses = [(x-3)**2]
    for _ in range(50):
        x = x - eta * 2*(x-3)
        losses.append((x-3)**2)
    ax.semilogy(losses, linewidth=2, label=f'eta={eta}')
ax.axhline(0.01, color='black', linestyle='--', alpha=0.5, label='target=0.01')
ax.set_xlabel('Iteracao')
ax.set_ylabel('Loss')
ax.set_title('Grid Search de Learning Rate')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('/tmp/ex1_lr_grid.png', dpi=100, bbox_inches='tight')
plt.close()
print("\nGrafico salvo!")

Melhor learning rate: 0.5
     eta |  Iteracoes |     Status
-----------------------------------
   0.001 |        100 |   NAO CONV
   0.010 |        100 |   NAO CONV
   0.050 |         38 |         OK
   0.100 |         18 |         OK
   0.300 |          5 |         OK
   0.500 |          1 |         OK



Grafico salvo!


### Exercicio 2: Comparar Otimizadores num Problema 2D
Usando os dados e funcoes auxiliares ja definidos no notebook (variaves `X`, `y`, `mse_loss`, `gradient`), implemente uma funcao unificada que roda qualquer otimizador e plote as curvas de convergencia de SGD, Momentum, e Adam no mesmo grafico.

**Dica:** crie uma funcao `optimize(method, X, y, eta, iterations)` que aceita `method` como string ("sgd", "momentum", "adam").

In [17]:
# EXERCICIO 2: Comparar Otimizadores
# Complete a funcao abaixo

def optimize(method, X, y, eta=0.01, iterations=200, beta1=0.9, beta2=0.999):
    """
    Otimizador unificado.
    method: 'sgd', 'momentum', ou 'adam'
    Retorna: lista de losses
    """
    w = np.array([0.0, 0.0])
    m = np.array([0.0, 0.0])  # 1o momento (momentum/adam)
    s = np.array([0.0, 0.0])  # 2o momento (adam)
    losses = [mse_loss(w, X, y)]

    for t in range(1, iterations + 1):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        grad = grad.ravel()

        if method == 'sgd':
            w = None  # TAREFA DO ALUNO
        elif method == 'momentum':
            m = None  # TAREFA DO ALUNO
            w = None  # TAREFA DO ALUNO
        elif method == 'adam':
            m = None  # TAREFA DO ALUNO
            s = None  # TAREFA DO ALUNO
            # Bias correction
            m_hat = None  # TAREFA DO ALUNO
            s_hat = None  # TAREFA DO ALUNO
            w = None  # TAREFA DO ALUNO

        losses.append(mse_loss(w, X, y))

    return losses

# Teste (descomentar apos implementar)
# for method in ['sgd', 'momentum', 'adam']:
#     losses = optimize(method, X, y)
#     plt.semilogy(losses, label=method)
# plt.legend(); plt.show()

In [18]:
# SOLUCAO Exercicio 2

def optimize(method, X, y, eta=0.01, iterations=200, beta1=0.9, beta2=0.999, epsilon=1e-8):
    w = np.array([0.0, 0.0])
    m = np.array([0.0, 0.0])
    s = np.array([0.0, 0.0])
    losses = [mse_loss(w, X, y)]

    for t in range(1, iterations + 1):
        idx = np.random.randint(0, len(y))
        grad = -2 * X[idx:idx+1].T @ (y[idx:idx+1] - X[idx:idx+1] @ w) / 1
        grad = grad.ravel()

        if method == 'sgd':
            w = w - eta * grad
        elif method == 'momentum':
            m = beta1 * m + (1 - beta1) * grad
            w = w - eta * m
        elif method == 'adam':
            m = beta1 * m + (1 - beta1) * grad
            s = beta2 * s + (1 - beta2) * grad**2
            m_hat = m / (1 - beta1**t)
            s_hat = s / (1 - beta2**t)
            w = w - eta * m_hat / (np.sqrt(s_hat) + epsilon)

        losses.append(mse_loss(w, X, y))

    return losses

np.random.seed(42)
fig, ax = plt.subplots(figsize=(12, 6))
colors = {'sgd': 'red', 'momentum': 'green', 'adam': 'blue'}

for method in ['sgd', 'momentum', 'adam']:
    np.random.seed(42)
    losses = optimize(method, X, y, eta=0.01)
    ax.semilogy(losses, linewidth=2, label=method.upper(), color=colors[method],
                alpha=0.8 if method != 'adam' else 1.0)

ax.set_xlabel('Iteracao')
ax.set_ylabel('MSE Loss')
ax.set_title('Comparacao: SGD vs Momentum vs Adam')
ax.legend(fontsize=12)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('/tmp/ex2_optimizers.png', dpi=100, bbox_inches='tight')
plt.close()
print("Comparacao completa!")
print("Adam converge mais rapido e suave que SGD e Momentum")

Comparacao completa!
Adam converge mais rapido e suave que SGD e Momentum


### Exercicio 3: Regularizacao L1 vs L2
Implemente regressao linear com regularizacao L1 e L2. Gere dados com 20 features, mas apenas 3 sao relevantes (os outros tem coeficiente zero). Compare os pesos aprendidos com L1 vs L2 e verifique que L1 produz pesos mais esparsos.

**Dica:** use `np.random.randn(100, 20)` para features e `w_true = [3, -2, 1.5, 0, 0, ..., 0]`.

In [19]:
# EXERCICIO 3: Regularizacao L1 vs L2
# Complete o codigo abaixo

np.random.seed(42)
n, d = 100, 20
X_reg = np.random.randn(n, d)
w_true_reg = np.zeros(d)
w_true_reg[:3] = [3, -2, 1.5]  # apenas 3 features relevantes
y_reg = X_reg @ w_true_reg + np.random.randn(n) * 0.5

def train_with_regularization(X, y, reg_type='l2', lam=0.1, eta=0.01, iterations=500):
    """
    reg_type: 'l2' ou 'l1'
    Retorna: pesos aprendidos
    """
    w = np.zeros(X.shape[1])

    for _ in range(iterations):
        grad_mse = -2 * X.T @ (y - X @ w) / len(y)

        if reg_type == 'l2':
            grad_reg = None  # TAREFA DO ALUNO: gradiente de lambda * ||w||^2
        elif reg_type == 'l1':
            grad_reg = None  # TAREFA DO ALUNO: gradiente de lambda * ||w||_1 (usar np.sign)

        w = w - eta * (grad_mse + grad_reg)

    return w

# Teste (descomentar apos implementar)
# w_l2 = train_with_regularization(X_reg, y_reg, 'l2', lam=0.1)
# w_l1 = train_with_regularization(X_reg, y_reg, 'l1', lam=0.1)
# print("L2 zeros:", np.sum(np.abs(w_l2) < 0.01))
# print("L1 zeros:", np.sum(np.abs(w_l1) < 0.01))

In [20]:
# SOLUCAO Exercicio 3

np.random.seed(42)
n, d = 100, 20
X_reg = np.random.randn(n, d)
w_true_reg = np.zeros(d)
w_true_reg[:3] = [3, -2, 1.5]
y_reg = X_reg @ w_true_reg + np.random.randn(n) * 0.5

def train_with_regularization(X, y, reg_type='l2', lam=0.1, eta=0.001, iterations=1000):
    w = np.zeros(X.shape[1])

    for _ in range(iterations):
        grad_mse = -2 * X.T @ (y - X @ w) / len(y)

        if reg_type == 'l2':
            grad_reg = 2 * lam * w
        elif reg_type == 'l1':
            grad_reg = lam * np.sign(w)
        else:
            grad_reg = np.zeros_like(w)

        w = w - eta * (grad_mse + grad_reg)

    return w

w_none = train_with_regularization(X_reg, y_reg, 'none', lam=0)
w_l2 = train_with_regularization(X_reg, y_reg, 'l2', lam=0.1)
w_l1 = train_with_regularization(X_reg, y_reg, 'l1', lam=0.1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, w, title in zip(axes, [w_none, w_l2, w_l1], ['Sem Regularizacao', 'L2 (Ridge)', 'L1 (Lasso)']):
    colors = ['green' if i < 3 else 'gray' for i in range(d)]
    ax.bar(range(d), w, color=colors, alpha=0.7)
    ax.bar(range(d), w_true_reg, color='none', edgecolor='red', linewidth=2, label='Verdadeiro')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Peso')
    ax.set_title(f'{title}\n(zeros: {np.sum(np.abs(w) < 0.05)})')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/ex3_regularization.png', dpi=100, bbox_inches='tight')
plt.close()

print("Resultado:")
print(f"  Sem reg - zeros (|w|<0.05): {np.sum(np.abs(w_none) < 0.05)}")
print(f"  L2 - zeros (|w|<0.05): {np.sum(np.abs(w_l2) < 0.05)}")
print(f"  L1 - zeros (|w|<0.05): {np.sum(np.abs(w_l1) < 0.05)}")
print("\nL1 produz mais zeros -> selecao automatica de features!")

Resultado:
  Sem reg - zeros (|w|<0.05): 6
  L2 - zeros (|w|<0.05): 6
  L1 - zeros (|w|<0.05): 9

L1 produz mais zeros -> selecao automatica de features!


## 16. Erros Comuns

### Erro 1: Usar MSE para Classificacao
**Sintoma:** loss diminui mas accuracy nao melhora; treinamento muito lento
**Causa:** MSE tem gradientes que encolhem quando sigmoid satura ($\hat{y} \approx 0$ ou $\approx 1$)
**Solucao:** Usar Cross-Entropy para classificacao binaria, Categorical CE para multiclasse

### Erro 2: Nao Normalizar Features antes de GD
**Sintoma:** convergencia muito lenta; curvas de loss com formato "zigzag"
**Causa:** features com escalas diferentes criam superficies de custo alongadas; GD oscila entre elas
**Solucao:** Normalizar cada feature para media 0, desvio 1: $x' = (x - \mu)/\sigma$

### Erro 3: Learning Rate Fixo em Todo o Treinamento
**Sintoma:** loss para de diminuir cedo (muito pequeno) ou diverge (muito grande)
**Causa:** o learning rate otimo muda conforme o treinamento avanca
**Solucao:** Usar learning rate scheduling (step decay, cosine annealing) ou otimizadores adaptativos (Adam)

### Erro 4: Confundir Minimo Local com Global
**Sintoma:** "loss parou de diminuir, entao convergiu para o otimo"
**Causa:** em problemas nao-convexos, pode ser um minimo local (ou saddle point)
**Solucao:** Treinar com multiplas inicializacoes; usar SGD (ruido ajuda a escapar); verificar se resultados sao consistentes

### Erro 5: Ignorar Regularizacao
**Sintoma:** training loss baixo mas test loss alto (gap crescente)
**Causa:** modelo memoriza dados de treino (overfitting)
**Solucao:** Adicionar L2 (weight decay) por padrao; tunar $\lambda$ via cross-validation

### Erro 6: Atribuir Vanishing Gradient a Falta de Dados
**Sintoma:** rede profunda nao aprende, aumentar dados nao ajuda
**Causa:** gradientes desaparecem antes de chegar as primeiras camadas
**Solucao:** Usar ReLU, batch normalization, skip connections. O problema eh arquitetural, nao de dados

### Erro 7: Batch Size Incorreto
**Sintoma:** treinamento instavel (batch muito pequeno) ou convergencia para minimo ruim (batch muito grande)
**Causa:** batch muito pequeno = estimativa muito ruidosa do gradiente; batch muito grande = pouco ruido = fica preso em minimos locais
**Solucao:** Comecar com batch_size = 32 ou 64; aumentar se GPU tem memoria; diminuir se loss nao converge

## 17. Resumo e Mapa Conceitual

### Hierarquia de Otimizadores

```
Problema: min L(w)
    |
    v
Gradient Descent (basico, usa todos os dados)
    |
    v
SGD (1 amostra aleatoria por vez)
    |
    +--------> Mini-batch GD (k amostras, padrao k=32-256)
    |
    v
+-- Momentum: acumula direcao (v = beta*v + grad)
|       |
|       v
|   +-- AdaGrad: learning rate adaptativo (s += grad^2)
|   |       |
|   |       v
|   +-- RMSprop: AdaGrad + decay exponencial (s = rho*s + (1-rho)*grad^2)
|       |
|       v
+-----> Adam = Momentum + RMSprop + correcao de vies  <-- DEFAULT
```

### Tabela de Conexoes

| Conceito deste notebook | Fundamento matematico | Notebook |
|-------------------------|----------------------|----------|
| Gradiente | Vetor de derivadas parciais | 0.4 |
| Convergencia exponencial | Series geometricas | 0.5 |
| MSE como custo | MLE gaussiano | 0.7 |
| Cross-entropy como custo | MLE de Bernoulli | 0.7 |
| Regularizacao L2 | Prior gaussiano (MAP) | 0.7 |
| Regularizacao L1 | Prior Laplaciano (MAP) | 0.7 |
| Ruido do SGD | Variancia amostral, LGN | 0.6 |
| Media movel exponencial | Series ponderadas | 0.5 |
| Hessiana e convexidade | Segundas derivadas | 0.4 |
| Solucao analitica (OLS) | Inversao de matrizes | 0.3 |

### Checklist de Competencias

Apos completar este modulo, voce deve ser capaz de:

- [ ] Formular qualquer problema de ML como otimizacao ($\min_w \mathcal{L}(w)$)
- [ ] Escolher a funcao de custo adequada (MSE, CE, Huber, Hinge)
- [ ] Implementar Gradient Descent do zero em NumPy
- [ ] Explicar por que learning rate eh o hiperparametro mais critico
- [ ] Comparar Batch GD, SGD e Mini-batch (trade-offs)
- [ ] Descrever como Momentum, AdaGrad, RMSprop evoluem ate Adam
- [ ] Aplicar regularizacao L1/L2 e explicar a conexao bayesiana
- [ ] Diagnosticar vanishing/exploding gradients e suas solucoes
- [ ] Treinar regressao linear e logistica do zero com GD

### Proximos Passos

- **1.1 (Estatistica Descritiva):** Medidas de tendencia central e dispersao que fundamentam a analise de dados antes de modelar
- **2.x (Modelos supervisionados):** Aplicar otimizacao a SVMs, arvores de decisao, redes neurais
- **3.x (Deep Learning):** Estrategias avancadas (batch normalization, learning rate scheduling, gradient clipping) para treinar redes muito profundas